# Multi-target: beberapa kolom target sekaligus (terpisah, rantai, label gabungan)
**Kapan dipakai:** submission meminta ≥ 2 kolom tebakan dari teks yang sama (mis. sentimen +
kategori, rating + direkomendasikan, beberapa skor esai). Target yang saling berhubungan bisa
saling membantu:

1. **terpisah**: satu model per target (pembanding).
2. **rantai** (classifier/regressor chain): target ke-2 memakai tebakan OOF target ke-1 sebagai
   fitur, dst. Urutan di `CHAIN_ORDER`, taruh target termudah di depan.
3. **label gabungan** (khusus kelas): kombinasi "rating=5, rekomen=1" jadi satu kelas, lalu
   peluang tiap target = jumlah peluang kombinasi yang memuatnya. Bagus kalau kombinasi sedikit.
4. **rata-rata** semua cara di atas.

Cara terbaik dipilih **per target** dari skor OOF. Fitur = TF-IDF kata + huruf. Untuk menambah
kolom tabel: jalankan `stacking_teks_tabular` per target dengan OOF dari sini di `EXTRA_OOF`.
**Data demo:** ulasan baju, target `rating` (5 kelas) + `recommended_ind` (2 kelas).

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.metrics import f1_score, mean_squared_error
from sklearn.model_selection import KFold

## Konfigurasi
`TASK = "angka"` untuk target regresi (semua target harus sejenis). Metrik dibuat "besar = baik"
(RMSE dibalik tanda) supaya pemilihan cara terbaik selalu `idxmax`.

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/multi_target"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample panitia
KEY_CSV = f"{OUTPUT_FOLDER}/kunci_demo.csv"                  # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                  # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]           # GANTI saat lomba
TARGETS = ["rating", "recommended_ind"]        # GANTI saat lomba: nama kolom = kolom sample
CHAIN_ORDER = TARGETS                          # alt: list(reversed(TARGETS))
TASK = "kelas"                                 # GANTI: "angka" untuk target regresi
MODEL_NAME = "tfidf_multi"
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    if TASK == "kelas":
        return f1_score(truth, pred.argmax(axis=1), average="macro")   # GANTI: metrik lomba
    return -np.sqrt(mean_squared_error(truth, pred[:, 0]))             # -RMSE: besar = baik

In [ ]:
def make_model():
    if TASK == "kelas":
        return SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
    return Ridge(alpha=2.0)
# alt kelas: LogisticRegression(C=4.0, max_iter=2000) → peluang lebih halus, 5–10x lebih lambat


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca data + hubungan antar target

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
CLASSES = {}
CODES = {}
for target in TARGETS:
    if TASK == "kelas":
        CLASSES[target], CODES[target] = np.unique(train[target].astype(str), return_inverse=True)
    else:
        CLASSES[target] = np.array([target])
        CODES[target] = train[target].astype(float).to_numpy()
    print(target, "| kelas/kolom:", len(CLASSES[target]))
if TASK == "kelas":
    relation = pd.crosstab(train[TARGETS[0]], train[TARGETS[1]], normalize="index")
else:
    relation = train[TARGETS].corr()
relation.round(3).to_csv(f"{OUTPUT_FOLDER}/hubungan_target.csv")
print(relation.round(3))

## Buat sample + kunci demo (HAPUS saat lomba)
`recommended_ind` ada di test baju → dipakai sebagai kunci demo (fitur hanya teks, tidak bocor).

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
for target in TARGETS:
    demo_sample[target] = train[target].mode()[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):      # jangan timpa sample panitia
    demo_sample.to_csv(SAMPLE_CSV, index=False)
rating_key = pd.read_csv(f"{DATA_FOLDER}/baju_wanita_test_labels.csv")
demo_key = test[[ID_COL, "recommended_ind"]].copy()
rating_by_id = pd.Series(rating_key["rating"].values, index=rating_key[ID_COL].values)
demo_key["rating"] = demo_key[ID_COL].map(rating_by_id).values
demo_key.to_csv(KEY_CSV, index=False)
print("sample + kunci demo ditulis:", demo_sample.shape, demo_key.shape)

## Lipatan + TF-IDF

In [ ]:
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train))
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
all_text = pd.concat([full_train, full_test], ignore_index=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=100000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
print("TF-IDF:", matrix_train.shape)

## OOF satu target
Kolom peluang memakai urutan kelas global, jadi kelas yang absen di satu lipatan tetap aman.

In [ ]:
def run_oof(X, X_test, labels, width):
    oof = np.zeros((X.shape[0], width), dtype="float32")
    test_pred = np.zeros((X_test.shape[0], width), dtype="float32")
    for idx_fit, idx_val in folds:
        fitted = make_model().fit(X[idx_fit], labels[idx_fit])
        if TASK == "kelas":
            columns = fitted.classes_
            oof[np.ix_(idx_val, columns)] = fitted.predict_proba(X[idx_val])
            test_pred[:, columns] += fitted.predict_proba(X_test) / N_FOLDS
        else:
            oof[idx_val, 0] = fitted.predict(X[idx_val])
            test_pred[:, 0] += fitted.predict(X_test) / N_FOLDS
    return oof, test_pred


METHODS = {"terpisah": {}, "rantai": {}}
START = time.time()
for target in TARGETS:
    METHODS["terpisah"][target] = run_oof(matrix_train, matrix_test, CODES[target],
                                          len(CLASSES[target]))
    print(target, "| terpisah:", round(metric(CODES[target], METHODS["terpisah"][target][0]), 4))
print("detik:", round(time.time() - START))

## Rantai
Tebakan OOF target sebelumnya dibakukan (rata-rata 0, simpangan 1) lalu ditempel ke TF-IDF.
Baris train memakai OOF (model yang tidak melihatnya), baris test memakai tebakan test.

In [ ]:
def standardize(block_train, block_test):
    mean = block_train.mean(axis=0)
    spread = block_train.std(axis=0) + 1e-6
    return (block_train - mean) / spread, (block_test - mean) / spread


chain_train = []
chain_test = []
for target in CHAIN_ORDER:
    X = matrix_train
    X_test = matrix_test
    if len(chain_train) > 0:
        X = hstack([matrix_train, csr_matrix(np.hstack(chain_train))]).tocsr()
        X_test = hstack([matrix_test, csr_matrix(np.hstack(chain_test))]).tocsr()
    result = run_oof(X, X_test, CODES[target], len(CLASSES[target]))
    METHODS["rantai"][target] = result
    extra_train, extra_test = standardize(result[0], result[1])
    chain_train.append(extra_train)
    chain_test.append(extra_test)
    print(target, "| rantai:", round(metric(CODES[target], result[0]), 4))

## Label gabungan (khusus kelas)
`combos` = satu baris per kombinasi, isinya kode kelas tiap target. Matriks 0/1 `to_target`
menjumlahkan peluang kombinasi ke kelas target.

In [ ]:
if TASK == "kelas":
    code_table = pd.DataFrame(CODES)
    joint_codes = code_table.groupby(TARGETS).ngroup()
    joint_codes = joint_codes.to_numpy()
    combos = code_table.groupby(joint_codes).first()
    joint_oof, joint_test = run_oof(matrix_train, matrix_test, joint_codes, len(combos))
    METHODS["label gabungan"] = {}
    for target in TARGETS:
        to_target = np.eye(len(CLASSES[target]))[combos[target].to_numpy()]
        METHODS["label gabungan"][target] = (joint_oof @ to_target, joint_test @ to_target)
    print("kombinasi:", len(combos), "| baris kombinasi terkecil:", np.bincount(joint_codes).min())

## Rata-rata + tabel banding

In [ ]:
average = {}
for target in TARGETS:
    oof_parts = []
    test_parts = []
    for name in METHODS:
        oof_parts.append(METHODS[name][target][0])
        test_parts.append(METHODS[name][target][1])
    average[target] = (np.mean(oof_parts, axis=0), np.mean(test_parts, axis=0))
METHODS["rata-rata"] = average
rows = []
for name in METHODS:
    row = {}
    for target in TARGETS:
        row[target] = round(metric(CODES[target], METHODS[name][target][0]), 4)
    rows.append(row)
table = pd.DataFrame(rows, index=list(METHODS))
table["rata-rata target"] = table[TARGETS].mean(axis=1)
table.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv")
BEST = {}
for target in TARGETS:
    BEST[target] = table[target].idxmax()    # alt: tulis manual, mis. "rantai"
print(table.round(4))
print("terbaik per target:", BEST)

In [ ]:
figure, ax = plt.subplots(figsize=(8, 4))
table[TARGETS].plot.bar(ax=ax, color=["#0072B2", "#E69F00", "#009E73", "#CC79A7"][: len(TARGETS)])
ax.set_ylabel("skor OOF (besar = baik)")
ax.set_ylim(table[TARGETS].min().min() - 0.02, table[TARGETS].max().max() + 0.01)
ax.tick_params(axis="x", rotation=0)
figure.savefig(f"{OUTPUT_FOLDER}/banding_metode.png", dpi=200, bbox_inches="tight")
plt.show()

## Simpan OOF per target (cara terbaik) + catat percobaan
Nama file `oof_<MODEL_NAME>_<target>.npy`. Kolom = kelas `np.unique` target itu (format
`ensemble_oof`), regresi n × 1.

In [ ]:
log_path = f"{OOF_FOLDER}/percobaan.csv"
for target in TARGETS:
    best_oof, best_test = METHODS[BEST[target]][target]
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}_{target}.npy", best_oof.astype("float32"))
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}_{target}.npy", best_test.astype("float32"))
    score = table.loc[BEST[target], target]
    log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                             "notebook": "multi_target", "model": f"{MODEL_NAME}_{target}",
                             "metrik": "f1_macro / -rmse", "skor_cv": score,
                             "detik": 0, "catatan": BEST[target]}])
    log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("disimpan:", TARGETS)

## Submission + cek

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
for target in TARGETS:
    best_test = METHODS[BEST[target]][target][1]
    values = best_test[:, 0]
    if TASK == "kelas":
        values = CLASSES[target][best_test.argmax(axis=1)]
    pred_by_id = pd.Series(values, index=test[ID_COL].values)
    submission[target] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
missing = written.isna().sum()
if len(written) != len(sample) or missing.sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
for target in TARGETS:
    counts = written[target].value_counts()
    print(target, counts.head(5).to_dict())
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)


def truth_codes(target):
    truth_by_id = pd.Series(key[target].values, index=key[ID_COL].values)
    truth = test[ID_COL].map(truth_by_id).to_numpy()
    if TASK == "kelas":
        return np.searchsorted(CLASSES[target], truth.astype(str))
    return truth.astype(float)


rows = []
for name in METHODS:
    row = {}
    for target in TARGETS:
        row[target] = round(metric(truth_codes(target), METHODS[name][target][1]), 4)
    rows.append(row)
print(pd.DataFrame(rows, index=list(METHODS)))